# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All elements, such as record sets, fields, and columns, are referenced by their `@id`, ensuring reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure the latest mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
We use `mlcroissant` to load the dataset metadata and inspect its structure and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata
metadata = dataset.metadata
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")
print(f"License: {metadata.license if hasattr(metadata, 'license') else 'N/A'}")
print(f"Citation: {metadata.citeAs if hasattr(metadata, 'citeAs') else 'N/A'}")

## 2. Data Overview
We review the record sets available in the dataset, their fields, and respective `@id`s for reference and extraction.

Below, we enumerate all record sets and their fields using only entity `@id`s.

In [ ]:
# List all record sets, fields, and field IDs

record_sets = list(dataset.list_record_sets())
print(f"Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- Record Set @id: {rs}")
    fields = dataset.list_fields(record_set=rs)
    print("  Fields and their @id's:")
    for field in fields:
        print(f"    - Field @id: {field}")

### Example: Previewing Records

Let's examine a few records from the tabular dataset. Replace `<record_set_id>` with the `@id` from the list above.

In [ ]:
# Preview first few records from a record set
# Select the main tabular record set (inspect above for correct @id)

# For this dataset, let's choose the first record set listed.
main_record_set_id = record_sets[0]

print(f"\nSample records from record set @id: {main_record_set_id}")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
We extract the data for each record set into a pandas DataFrame for further analysis. All lookups use the respective `@id`s.

Below, we loop through each available record set and load the data.

In [ ]:
# Extract each record set (@id) as a DataFrame

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set @id: {record_set_id}")

# Display the columns of the main record set
main_df = dataframes[main_record_set_id]
print(f"\nColumns (@id) in main record set {main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
We now process the tabular data, performing typical EDA operations: filtering, normalization, and grouping using column `@id`s only.

#### 1. Select a Numeric Field to Filter, Normalize, and Group

Inspect the DataFrame columns from above. Suppose a numeric field (e.g., patient age or an interval) has the `@id` `'age_at_second_crc'` and grouping would be by `'sex'` (replace with actual `@id`s as needed).

In [ ]:
# Replace the following @id's with those from your columns if different.
numeric_field_id = None
possible_numeric_fields = [col for col in main_df.columns if main_df[col].dtype in [int, float, 'int64', 'float64']]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use the first detected numeric field

if numeric_field_id is not None:
    print(f"Using numeric field @id: {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype != object else 10
    # Filter
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a categorical field
    # Choose a likely categorical field (e.g. 'sex', 'msi_status', etc.)
    possible_group_fields = [col for col in main_df.columns if main_df[col].nunique() <= 10 and main_df[col].dtype == object]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in main record set.")

## 5. Visualization
Visualize numeric distributions and group comparisons. Use only column `@id`s to reference variables in plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # Boxplot by group
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated loading, navigating, and analyzing a FAIR² dataset package via the Croissant schema using the `mlcroissant` library. All data extraction and referencing used the canonical `@id` fields as recommended for reproducible, transparent research. 

Further domain-specific analysis and more advanced visualization can be built upon these foundations.